# Import Library

In [1]:
import pandas as pd
import re

from google.colab import drive

# Load Data

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
dataset_path = "/content/drive/MyDrive/Dataset/Skripsi/dataset_labeled.csv"

In [4]:
df = pd.read_csv(dataset_path)

# Preprocessing

In [5]:
# fungsi case folding

def case_folding(text):
    text = str(text).lower()
    return text

In [6]:
# fungsi cleaning

def cleaning(text):
    # menghapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # menghapus mention dan hashtag
    text = re.sub(r'@\w+|#\w+', '', text)

    # menghapus angka
    text = re.sub(r'\d+', '', text)

    # menghapus karakter selain huruf
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # menghapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [7]:
# kamus normalisasi

normalization_dict = {
    'gk': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'tdk': 'tidak', 'bgt': 'banget',
    'apk': 'aplikasi', 'udh': 'sudah', 'sm': 'sama', 'yg': 'yang', 'dr': 'dari',
    'tp': 'tapi', 'trs': 'terus', 'krn': 'karena', 'dpt': 'dapat', 'jg': 'juga',
    'blm': 'belum', 'cm': 'cuma', 'aja': 'saja'
}

In [8]:
# fungsi normalisasi

def normalization(text):
    words = text.split()
    normalized_words = []

    for word in words:
        if word in normalization_dict:
            normalized_words.append(normalization_dict[word])
        else:
            normalized_words.append(word)

    return ' '.join(normalized_words)

In [9]:
# menggabungkan preprocessing

def preprocessing(text):
    text = case_folding(text)
    text = cleaning(text)
    text = normalization(text)
    return text

In [10]:
# menerapkan preprocessing
df['Preprocessed'] = df['ULASAN'].apply(preprocessing)

# menampilkan hasil
print(df[['ULASAN', 'Preprocessed']].head())

                                              ULASAN  \
0  Sudah sekali kembalikan uang yang sudah terpot...   
1  ini aplikasi dana apaan sih?? mau pindah akun ...   
2  Mau login dana nomor lama nya hilang terus sur...   
3  Bintang 2 dulu, karena baru juga download.. lo...   
4  kecewa sih sama aplikasi ini padahal biasanya ...   

                                        Preprocessed  
0  sudah sekali kembalikan uang yang sudah terpot...  
1  ini aplikasi dana apaan sih mau pindah akun ma...  
2  mau login dana nomor lama nya hilang terus sur...  
3  bintang dulu karena baru juga download login a...  
4  kecewa sih sama aplikasi ini padahal biasanya ...  


In [11]:
# menyimpan hasil preprocessing
df.to_csv('dataset_preprocessed.csv', index=False)

# Implementasi BERT

## Persiapan Lingkungan dan Library

In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm

## Parameter dan Konfigurasi Model

In [13]:
MAX_LEN = 128
BATCH_SIZE = 16
LEARNING_RATE = 1e-5
EPOCHS = 7

MODEL_NAME = 'indobenchmark/indobert-base-p1'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(DEVICE)

cuda


## Load Dataset dan Tokenisasi BERT

In [14]:
# membaca dataset
df = pd.read_csv('dataset_preprocessed.csv')

# mengambil teks
texts = df['Preprocessed'].values

# mengambil seluruh label aspek-sentimen
labels = df[[
    'APLIKASI_POSITIF',
    'APLIKASI_NEGATIF',
    'INTERFACE_POSITIF',
    'INTERFACE_NEGATIF',
    'LAYANAN_POSITIF',
    'LAYANAN_NEGATIF',
    'KEAMANAN_POSITIF',
    'KEAMANAN_NEGATIF'
]].values

print(df.head())

                                              ULASAN  APLIKASI_POSITIF  \
0  Sudah sekali kembalikan uang yang sudah terpot...                 0   
1  ini aplikasi dana apaan sih?? mau pindah akun ...                 0   
2  Mau login dana nomor lama nya hilang terus sur...                 0   
3  Bintang 2 dulu, karena baru juga download.. lo...                 0   
4  kecewa sih sama aplikasi ini padahal biasanya ...                 0   

   APLIKASI_NEGATIF  INTERFACE_POSITIF  INTERFACE_NEGATIF  LAYANAN_POSITIF  \
0                 1                  0                  1                0   
1                 1                  0                  0                0   
2                 1                  0                  0                0   
3                 1                  0                  0                0   
4                 1                  0                  0                0   

   LAYANAN_NEGATIF  KEAMANAN_POSITIF  KEAMANAN_NEGATIF  \
0                1          

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [16]:
sample = tokenizer(
    texts[0],
    padding='max_length',
    truncation=True,
    max_length=MAX_LEN,
    return_tensors='pt'
)

print(sample)

{'input_ids': tensor([[    2,   259,   684, 17853,   988,    34,   259, 27755,   356,  1869,
          9109,   440,  1736,  4914,  6353,  1473,  1339,    92, 11976,  1489,
          2088,  1107,  1339,   295,  3089,    26,   300, 11565,  1396,   271,
          1505,  3266,  6234,   176,  1505,  1740,   912,   186, 11202,  2054,
           804,  1489,   166,    26,  1861,    17, 25845, 30356,  3108,  1489,
           166,    26,  5120,   804,  1339,  1489,   166,    26,  2697,    79,
          2581,   158, 29510,   422,  4068,  1339,   245,   186,   500,   377,
          2592,  2732,  1500,  1505,   168,     3,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

## Pembuatan Dataset dan DataLoader

In [17]:
# pembagian dataset

# Identify rows in labels that contain NaN values
# Convert labels to a pandas DataFrame to easily detect NaNs across rows,
# as the labels array has dtype=object and might contain mixed types.
labels_df = pd.DataFrame(labels)
nan_rows_mask = labels_df.isna().any(axis=1)

# Filter out rows with NaN values from both texts and labels
texts_cleaned = texts[~nan_rows_mask]
labels_cleaned = labels[~nan_rows_mask].astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    texts_cleaned,
    labels_cleaned,
    test_size=0.2,
    random_state=42
)

In [18]:
# pembuatan kelas dataset

class SentimentDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label.astype(np.float32), dtype=torch.float)
        }

In [19]:
# pembuatan dataloader

train_dataset = SentimentDataset(
    X_train,
    y_train,
    tokenizer,
    MAX_LEN
)

test_dataset = SentimentDataset(
    X_test,
    y_test,
    tokenizer,
    MAX_LEN
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)

## Arsitektur Model dan Optimasi

In [20]:
# modifikasi arsitektur model

class IndoBERTClassifier(nn.Module):

    def __init__(self, n_classes):
        super(IndoBERTClassifier, self).__init__()

        self.bert = AutoModel.from_pretrained(MODEL_NAME)

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled_output = outputs.last_hidden_state[:, 0]

        output = self.dropout(pooled_output)

        return self.fc(output)

In [21]:
# optimasi model

model = IndoBERTClassifier(n_classes=labels.shape[1])
model = model.to(DEVICE)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

loss_fn = nn.BCEWithLogitsLoss().to(DEVICE)

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Fungsi Pelatihan dan Evaluasi Model

In [22]:
# fungsi pelatihan model

def train_model(model, data_loader, loss_fn, optimizer, device, scheduler):

    model.train()

    losses = []
    correct_predictions = 0
    total_labels = 0

    for batch in tqdm(data_loader):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # reset gradient
        optimizer.zero_grad()

        # forward
        outputs = model(input_ids, attention_mask)

        # hitung loss
        loss = loss_fn(outputs, labels)

        # prediksi multi-label
        preds = torch.sigmoid(outputs)
        preds = (preds > THRESHOLD).float()

        # hitung accuracy
        correct_predictions += (preds == labels).sum().item()
        total_labels += labels.numel()

        # simpan loss
        losses.append(loss.item())

        # backward
        loss.backward()

        # update parameter
        optimizer.step()
        scheduler.step()

    return (
        correct_predictions / total_labels,
        np.mean(losses)
    )

In [23]:
# fungsi evaluasi model

def eval_model(model, data_loader, loss_fn, device):

    model.eval()

    losses = []
    correct_predictions = 0
    total_labels = 0

    predictions = []
    real_values = []

    with torch.no_grad():

        for batch in tqdm(data_loader):

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # forward
            outputs = model(input_ids, attention_mask)

            # hitung loss
            loss = loss_fn(outputs, labels)

            # sigmoid multi-label
            preds = torch.sigmoid(outputs)
            preds = (preds > THRESHOLD).float()

            # hitung accuracy
            correct_predictions += (preds == labels).sum().item()
            total_labels += labels.numel()

            # simpan loss
            losses.append(loss.item())

            # simpan prediksi
            predictions.extend(preds.cpu().numpy())
            real_values.extend(labels.cpu().numpy())

    return (
        correct_predictions / total_labels,
        np.mean(losses),
        predictions,
        real_values
    )

## Proses Pelatihan dan Penyimpanan Model

In [24]:
# proses pelatihan model

history = {
    'train_loss': [],
    'val_loss': [],
    'train_acc': [],
    'val_acc': []
}

best_accuracy = 0

for epoch in range(EPOCHS):

    print(f'\nEpoch {epoch + 1}/{EPOCHS}')
    print('-' * 50)

    # =========================
    # TRAINING
    # =========================

    model.train()

    train_losses = []
    train_correct = 0
    total_train = 0

    for batch in tqdm(train_loader):

        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask)

        loss = loss_fn(outputs, labels)

        loss.backward()

        optimizer.step()
        scheduler.step()

        train_losses.append(loss.item())

        # prediksi multi-label
        preds = torch.sigmoid(outputs)
        preds = (preds > 0.5).float()

        train_correct += (preds == labels).sum().item()
        total_train += labels.numel()

    train_accuracy = train_correct / total_train
    train_loss = np.mean(train_losses)

    # =========================
    # VALIDASI / EVALUASI
    # =========================

    model.eval()

    val_losses = []
    val_correct = 0
    total_val = 0

    with torch.no_grad():

        for batch in tqdm(test_loader):

            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids, attention_mask)

            loss = loss_fn(outputs, labels)

            val_losses.append(loss.item())

            preds = torch.sigmoid(outputs)
            preds = (preds > 0.5).float()

            val_correct += (preds == labels).sum().item()
            total_val += labels.numel()

    val_accuracy = val_correct / total_val
    val_loss = np.mean(val_losses)

    # simpan history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_accuracy)
    history['val_acc'].append(val_accuracy)

    # tampilkan hasil epoch
    print(f'Train Loss     : {train_loss:.4f}')
    print(f'Train Accuracy : {train_accuracy:.4f}')
    print(f'Val Loss       : {val_loss:.4f}')
    print(f'Val Accuracy   : {val_accuracy:.4f}')

    # =========================
    # SIMPAN MODEL TERBAIK
    # =========================

    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            'best_model_indobert.pt'
        )

        print('Model terbaik berhasil disimpan')

print('\nTraining selesai')


Epoch 1/7
--------------------------------------------------


  0%|          | 0/250 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

100%|██████████| 63/63 [00:10<00:00,  6.25it/s]


Train Loss     : 0.3432
Train Accuracy : 0.8358
Val Loss       : 0.2838
Val Accuracy   : 0.8639
Model terbaik berhasil disimpan

Epoch 2/7
--------------------------------------------------


...

Train Loss     : 0.1919
Train Accuracy : 0.9191
Val Loss       : 0.2802
Val Accuracy   : 0.8771

Epoch 6/7
--------------------------------------------------


100%|██████████| 63/63 [00:09<00:00,  6.76it/s]


Train Loss     : 0.1738
Train Accuracy : 0.9306
Val Loss       : 0.2856
Val Accuracy   : 0.8769

Epoch 7/7
--------------------------------------------------


100%|██████████| 63/63 [00:09<00:00,  6.75it/s]

Train Loss     : 0.1596
Train Accuracy : 0.9367
Val Loss       : 0.2883
Val Accuracy   : 0.8745

Training selesai


## Evaluasi Model

In [25]:
# mode evaluasi
model.eval()

predictions = []
actual_labels = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        # forward
        outputs = model(input_ids, attention_mask)

        # sigmoid untuk multi-label classification
        preds = torch.sigmoid(outputs)

        # threshold 0.5
        preds = (preds > 0.5).float()

        # simpan prediksi dan label asli
        predictions.extend(preds.cpu().numpy())
        actual_labels.extend(labels.cpu().numpy())

# ubah ke numpy array
predictions = np.array(predictions)
actual_labels = np.array(actual_labels)

# =========================
# PERHITUNGAN METRIK
# =========================

accuracy = np.mean(predictions == actual_labels)

precision = precision_score(
    actual_labels,
    predictions,
    average='micro'
)

recall = recall_score(
    actual_labels,
    predictions,
    average='micro'
)

f1 = f1_score(
    actual_labels,
    predictions,
    average='micro'
)

# =========================
# HASIL EVALUASI
# =========================

print('=' * 50)
print('HASIL EVALUASI MODEL')
print('=' * 50)

print(f'Accuracy  : {accuracy:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1-Score  : {f1:.4f}')

# =========================
# CLASSIFICATION REPORT
# =========================

print('\nClassification Report')

print(classification_report(
    actual_labels,
    predictions,
    target_names=[
        'APLIKASI_POSITIF',
        'APLIKASI_NEGATIF',
        'INTERFACE_POSITIF',
        'INTERFACE_NEGATIF',
        'LAYANAN_POSITIF',
        'LAYANAN_NEGATIF',
        'KEAMANAN_POSITIF',
        'KEAMANAN_NEGATIF'
    ]
))

100%|██████████| 63/63 [00:09<00:00,  6.76it/s]

HASIL EVALUASI MODEL
Accuracy  : 0.8745
Precision : 0.7709
Recall    : 0.7716
F1-Score  : 0.7713

Classification Report
                   precision    recall  f1-score   support

 APLIKASI_POSITIF       0.88      0.79      0.83       151
 APLIKASI_NEGATIF       0.74      0.83      0.78       606
INTERFACE_POSITIF       0.61      0.59      0.60        37
INTERFACE_NEGATIF       0.68      0.58      0.63       320
  LAYANAN_POSITIF       0.76      0.72      0.74        57
  LAYANAN_NEGATIF       0.80      0.83      0.81       664
 KEAMANAN_POSITIF       0.65      0.62      0.63        21
 KEAMANAN_NEGATIF       0.84      0.77      0.81       338

        micro avg       0.77      0.77      0.77      2194
        macro avg       0.75      0.72      0.73      2194
     weighted avg       0.77      0.77      0.77      2194
      samples avg       0.78      0.80      0.75      2194




/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
